# Calculating metrics

### Fidelity

$$Fidelity+^{prob}=\frac{1}{N}\sum_{i=1}^{N}(f(G_{i})_{y_{i}}-f(G_{i}^{\overline{m}_{i}})_{y_{i}})$$

In [1]:
%cd ..
%pwd

In [2]:
import torch
import re
import logging
import json
import torch_geometric.transforms as T
from pathlib import Path
from src.data.text_graph_dataset_ondisk import TextGraphDatasetOnDisk
from src.models.graph_classification.gnn import DiffPoolMinCut
from src.models.graph_explainability.explainability_metrics import calculate_fidelity
from src.utils.general_utils import load_config

[nltk_data] Downloading package punkt to /home/trdp/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
# ==========================================
# 1. Configuration
# ==========================================
MODEL_PATH = "models/IMDB/best_model_DiffPoolMinCut_2025.11.24_23.28.59_final_lr0.0009615826222685446_hd_64_bs32_dec0.0624919749252487_lk0.01_en0.001_rc0.0_ct0.1_bl0.01_rp1.0_l20.009909917114455734_rep00.pth"
DATASET = "IMDB"
ROOT = f"data/datasets/{DATASET}"
LANG = "english"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Config for fixed dimensions (like HIDDEN_DIM)
# Assuming you run this from the project root
CONFIG = load_config(LANG, "src/utils/config.json")

def parse_hyperparameters(model_path, config):
    """
    Parses hyperparameters.
    Crucially fixes the 'hidden_dim' vs 'inner_dim' mismatch.
    """
    filename = Path(model_path).name
    params = {}

    # 1. Parse dynamic params from filename
    # Note: 'hd' in filename corresponds to 'inner_dim' in the training script logic
    patterns = {
        "inner_channels": r"_hd_(\d+)_",      # Mapped 'hd' to inner_channels
        "decrease_proportion": r"_dec([\d\.]+)_",
    }

    for key, pattern in patterns.items():
        match = re.search(pattern, filename)
        if match:
            val = float(match.group(1))
            params[key] = int(val) if val.is_integer() else val

    # 2. Load static params from Config (The Source of Truth for hidden_dim)
    defaults = {
        "max_num_nodes": config.get("NUM_NODES", 1000),
        "in_channels": config.get("NODE_FEATURE_DIM", 100),
        "out_channels": 2, # Fixed for IMDB
        "hidden_channels": config.get("HIDDEN_DIM", 100), # This fixes the size 100 mismatch
        "softmax_assign": True
    }

    final_params = {**defaults, **params}
    print(f"Final Model Params: {final_params}")
    return final_params

In [4]:
# ==========================================
# 2. Model Initialization
# ==========================================
print("Initializing Model...")
params = parse_hyperparameters(MODEL_PATH, CONFIG)

best_diffpool_model = DiffPoolMinCut(
    max_num_nodes=params['max_num_nodes'],
    in_channels=params['in_channels'],
    hidden_channels=params['hidden_channels'], # Will be 100
    out_channels=params['out_channels'],
    inner_channels=params['inner_channels'],   # Will be 64 (from filename 'hd')
    softmax_assign=params['softmax_assign'],
    decrease_proportion=params['decrease_proportion'],
)

# Load Weights
print(f"Loading weights from {MODEL_PATH}")
weights = torch.load(MODEL_PATH, map_location=DEVICE)
best_diffpool_model.load_state_dict(weights)
best_diffpool_model = best_diffpool_model.to(DEVICE)
best_diffpool_model.eval()

print("Model loaded successfully!")

/tmp/ipykernel_39570/2534661162.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights = torch.load(MODEL_PATH, map_location=DEVICE)


Initializing Model...
Final Model Params: {'max_num_nodes': 1000, 'in_channels': 100, 'out_channels': 2, 'hidden_channels': 100, 'softmax_assign': True, 'inner_channels': 64, 'decrease_proportion': 0.0624919749252487}
Loading weights from models/IMDB/best_model_DiffPoolMinCut_2025.11.24_23.28.59_final_lr0.0009615826222685446_hd_64_bs32_dec0.0624919749252487_lk0.01_en0.001_rc0.0_ct0.1_bl0.01_rp1.0_l20.009909917114455734_rep00.pth
Model loaded successfully!


In [5]:
import random

print("Loading Test Data...")

transform = T.Compose([
    T.ToDense(num_nodes=params['max_num_nodes']),
])

tgd_test = TextGraphDatasetOnDisk(
    root=ROOT,
    split="test",
    node_feature_size=params['in_channels'],
    transform=transform,
    max_num_nodes=params['max_num_nodes'],
    lang=LANG
)

# Select a sample
SAMPLE_IDX = random.randint(0, len(tgd_test) - 1)
data_sample = tgd_test[SAMPLE_IDX].to(DEVICE)

# Handle case where doc_id might be a list or a tensor
if isinstance(data_sample.doc_id, list):
    data_sample_id = data_sample.doc_id[0]
else:
    data_sample_id = data_sample.doc_id.item()

print(f"Processing Sample ID: {data_sample_id}")

# Dataset returns single items [N, F], Model expects batches [B, N, F]

# 1. Handle Features (x)
if isinstance(data_sample.x, list):
    x = torch.tensor(data_sample.x, device=DEVICE).unsqueeze(0)
else:
    x = data_sample.x.unsqueeze(0)

# 2. Handle Adjacency (adj)
if isinstance(data_sample.adj, list):
    adj = torch.tensor(data_sample.adj, device=DEVICE).unsqueeze(0)
else:
    adj = data_sample.adj.unsqueeze(0)

# 3. Handle Mask (mask)
if isinstance(data_sample.mask, list):
    mask = torch.tensor(data_sample.mask, device=DEVICE).unsqueeze(0)
else:
    mask = data_sample.mask.unsqueeze(0)

# 4. Handle Label (y)
if isinstance(data_sample.y, list):
    y = torch.tensor(data_sample.y, device=DEVICE)
else:
    y = data_sample.y

# Ensure y has shape [Batch_Size] -> [1]
if y.dim() == 0:
    y = y.unsqueeze(0)

Loading Test Data...
Processing Sample ID: 107


In [6]:
# ==========================================
# 4. Inference & Explanation Generation
# ==========================================

# Run Forward Pass with Debug to get internals
with torch.no_grad():
    prediction = best_diffpool_model(x, adj, mask=mask, debug=True)

# Unpack results
y_logits, _, _, _, _, _, (s01, s12) = prediction
y_pred_prob = torch.softmax(y_logits, dim=1)
predicted_class = torch.argmax(y_logits, dim=1).item()

print(f"Prediction: {y_pred_prob.cpu().squeeze().tolist()} | Class: {predicted_class}")

# Generate Explanation (Top-K Nodes in Layer 1 clusters)
s01_squeezed = s01.squeeze(0) 
k = 10
row_max_vals, _ = torch.max(s01_squeezed, dim=1) 
topk_vals, topk_indices = torch.topk(row_max_vals, k)
explanation_nodes = topk_indices.tolist()

print(f"Explanation Nodes: {explanation_nodes}")

Prediction: [0.7011680603027344, 0.298831969499588] | Class: 0
Explanation Nodes: [35, 34, 12, 3, 77, 13, 9, 62, 47, 10]


In [7]:
# ==========================================
# 5. Fidelity Calculation
# ==========================================
print("Calculating Fidelity...")

fid_plus, fid_minus = calculate_fidelity(
    model=best_diffpool_model,
    x=x,
    explanation=explanation_nodes, 
    level=0, 
    adj=adj,
    mask=mask
)

print("\n" + "="*30)
print(f"RESULTS FOR SAMPLE {data_sample_id}")
print("="*30)
print(f"Target Class: {predicted_class}")
print(f"Fidelity+ (Necessity):   {fid_plus:.4f} (Higher is better)")
print(f"Fidelity- (Sufficiency): {fid_minus:.4f} (Lower is better)")
print("="*30)

Calculating Fidelity...

RESULTS FOR SAMPLE 107
Target Class: 0
Fidelity+ (Necessity):   -0.1083 (Higher is better)
Fidelity- (Sufficiency): -0.2988 (Lower is better)
